# Lecture 11 — Nested `for` Loops
### EES 3350/5350 · Python in Earth Science · Monday, September 28, 2026

**Learning objectives for today**
- explain, step by step, what happens when one loop sits inside another (and predict the output before running it)
- write a nested `for` loop that runs a sweep for *every* item in a collection
- loop through nested data: a dictionary of dictionaries and a dictionary of lists
- build a dictionary of lists with a nested loop (and know where the empty list has to be created)
- recognize and fix three classic nested-loop bugs
- get a first look at `continue` and `break` inside nested loops (we go deeper on Friday)

---
## Part 0 : set up
Each lecture starts a fresh Python session, so let's rebuild what we need. Today we combine the two datasets you know best: **slope stability** from Lab 3 (and Lectures 09–10) and the **Calwood Fire gullies** from Lab 2.

First, `factor_of_safety()` and `interpret_stability()`, exactly as we left them in Lecture 10:

In [1]:
import math

def factor_of_safety(cohesion, soil_depth, soil_density, 
                     slope_deg, friction_deg, rw,  gravity=9.81, water_density=1000):
    """
    Compute the infinite slope Factor of Safety (FS).

    Parameters
    ----------
    cohesion : float
        Cr + Cs : cohesion - bonds holding roots (r) and soil (s) together (Pa)
    soil_depth : float
        depth of soil perpendicular to the slope (m)
    soil_density : float
        density of wet soil (kg/m^3)
    gravity : float
        gravitational acceleration (m/s^2)
    slope_deg : float
        slope angle (angle of the soil surface from horizontal) (degrees)
    friction_deg : float
        internal friction angle of the soil (degrees)
    rw : float
        relative wetness - ratio of groundwater depth to soil depth (1 = fully saturated) (0-1)
    water_density : float
        density of water (kg/m^3)

    Returns
    -------
    float, or None if slope_deg == 0 (undefined, would divide by 0)
    """
    # converting degrees to radians
    theta = math.radians(slope_deg)
    phi = math.radians(friction_deg)

    # A
    denom_A = soil_depth * soil_density * gravity * math.sin(theta)
    A = cohesion/denom_A
    # B
    parenth_B = (1 - ((rw * water_density) / soil_density))
    numerat_B = math.cos(theta) * math.tan(phi) * parenth_B
    B = numerat_B / math.sin(theta)

    FS = A + B
    
    return FS

def interpret_stability(fs):
    """Classify a Factor of Safety value into a stability category."""
    if fs < 1.0:
        return "Unstable"
    elif fs < 1.3:
        return "Marginally stable"
    else:
        return "Stable"

Next, the four soil scenarios from Lecture 10 (including the `"extreme"` one you added in Exercise 1). This time we've taken `rw` **out** of each scenario, because relative wetness is what we're about to sweep.

In [2]:
soil_scenarios = {
    "low":     {"cohesion": 0,     "soil_depth": 0.2, "soil_density": 2000,
                "slope_deg": 5,  "friction_deg": 25},
    "medium":  {"cohesion": 8500,  "soil_depth": 1.0, "soil_density": 2000,
                "slope_deg": 25, "friction_deg": 32},
    "high":    {"cohesion": 17000, "soil_depth": 2.5, "soil_density": 2000,
                "slope_deg": 45, "friction_deg": 40},
    "extreme": {"cohesion": 20000, "soil_depth": 3.0, "soil_density": 2000,
                "slope_deg": 40, "friction_deg": 38},
}

And the four burned gullies from Lab 2, bundled into one nested dictionary. It's the same data you typed into lists in Lab 2 (area, burn severity, and infiltration rates before and 3 months after the fire), plus the three storms from Lab 2, Exercise 18. We're also wrapping the Caine (1980) debris-flow threshold in a function so we can reuse it.

In [3]:
gully_profiles = {
    "Heil-Ranch-gully1": {"area_km2": 0.1538, "burn_severity": "severe",
                          "normal_infiltration": 16, "postfire_infiltration": 2.25},
    "Heil-Ranch-gully2": {"area_km2": 0.2523, "burn_severity": "moderate",
                          "normal_infiltration": 16, "postfire_infiltration": 11.77},
    "Geer-Canyon":       {"area_km2": 7.325,  "burn_severity": "severe",
                          "normal_infiltration": 22, "postfire_infiltration": 3.5},
    "Central-Gulch":     {"area_km2": 0.1515, "burn_severity": "moderate",
                          "normal_infiltration": 16, "postfire_infiltration": 4.0},
}

ET = 3                                      # mm/hr, evapotranspiration assumed in Lab 2

# the three (hypothetical) storms from Lab 2, each as (intensity in mm/hr, duration in hr)
storms = [(30, 1), (18, 0.2), (45, 3)]

def caine_threshold(duration_hr):
    """Caine (1980) threshold rainfall intensity (mm/hr) for a storm lasting duration_hr hours."""
    return 14.82 * duration_hr ** (-0.39)

Here is what you can already do with **one** loop. Remember Lab 2, Exercise 8, where you calculated the drop in infiltration for each gully one at a time? With a loop:

In [4]:
for gully, props in gully_profiles.items():
    drop = props["normal_infiltration"] - props["postfire_infiltration"]
    print(f"{gully:>18}: infiltration dropped by {drop:5.2f} mm/hr")

 Heil-Ranch-gully1: infiltration dropped by 13.75 mm/hr
 Heil-Ranch-gully2: infiltration dropped by  4.23 mm/hr
       Geer-Canyon: infiltration dropped by 18.50 mm/hr
     Central-Gulch: infiltration dropped by 12.00 mm/hr


<div class="alert alert-success">

**Tip**: `{drop:5.2f}` packs two format specs into one: `5` is the *width* (pad to 5 characters, so the numbers line up) and `.2f` is the *precision* (2 decimal places). It's the same idea as `{name:>6}` and `{fs:.3f}` from Lecture 10, just together.

</div>

---
## Part 1 : reading a nested loop

That was **one** loop. Now we'll put a loop *inside* another loop. Before we bring the science back, here's the bare skeleton, with nothing but counting:

In [5]:
for i in range(3):
    for j in range(2):
        print(f"outer i = {i}, inner j = {j}")

outer i = 0, inner j = 0
outer i = 0, inner j = 1
outer i = 1, inner j = 0
outer i = 1, inner j = 1
outer i = 2, inner j = 0
outer i = 2, inner j = 1


| **pass** | **`i` (outer)** | **`j` (inner)** | **what it means** |
|-:|:-:|:-:|:--|
| 1 | 0 | 0 | outer takes its 1st step; inner starts |
| 2 | 0 | 1 | inner takes its 2nd step |
| 3 | 1 | 0 | inner is *finished*, so outer takes its 2nd step and inner **restarts** |
| 4 | 1 | 1 | |
| 5 | 2 | 0 | outer takes its 3rd step; inner restarts again |
| 6 | 2 | 1 | both loops are done |

**Three rules to hold on to:**
1. The outer loop takes **one** step.
2. The inner loop runs **all the way through**.
3. The outer loop takes its next step, and the inner loop **starts over**.

Think of a clock: the minute hand (inner) goes all the way around before the hour hand (outer) moves one step. The number of passes through the innermost line is `(outer passes) × (inner passes)`.

<div class="alert alert-success">

**Tip**: indentation decides which loop a line belongs to. A line indented one level belongs to the outer loop and runs once per outer pass. A line indented two levels belongs to the inner loop and runs once per inner pass.

</div>

#### Another way to see it: rows and columns
Nested loops are perfect for filling in a grid. The **outer** loop walks down the *rows*, and the **inner** loop walks across the *columns*. To draw one, we need a new trick with `print()`:

In [6]:
for row in range(3):                  # outer loop: one pass per row
    for col in range(4):              # inner loop: one pass per column
        print("*", end=" ")           # print a star, but stay on the same line
    print()                           # inner loop is done: now start a new line

* * * * 
* * * * 
* * * * 


<div class="alert alert-success">

**Tip**: `end=" "`

Normally `print()` finishes by starting a new line. `print("*", end=" ")` swaps that new line for a space, so the next `print` continues on the *same* line. The bare `print()` at the end of each row prints nothing but the new line. Notice where it's indented: at the **outer** loop's level, so it runs once per row.

</div>

<div class="alert alert-info">

### Exercise 1 : predict, then run
Read the code below. **Before you run it**, write down:

**(a)** how many lines it will print, and

**(b)** where the `--- finished ---` lines will appear.

Then run the cell to check your prediction.

**(c)** Now indent the `print(f"--- finished ...")` line one level deeper (so it lines up with the other `print`). Predict again: how many lines now? Run it and see.

</div>

In [ ]:
for gully in ["Geer-Canyon", "Central-Gulch", "Heil-Ranch-gully1"]:
    for storm_number in [1, 2]:
        print(f"{gully} | storm {storm_number}")
    print(f"--- finished {gully} ---")

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
for gully in ["Geer-Canyon", "Central-Gulch", "Heil-Ranch-gully1"]:
    for storm_number in [1, 2]:
        print(f"{gully} | storm {storm_number}")
    print(f"--- finished {gully} ---")
```
---
```
Geer-Canyon | storm 1
Geer-Canyon | storm 2
--- finished Geer-Canyon ---
Central-Gulch | storm 1
Central-Gulch | storm 2
--- finished Central-Gulch ---
Heil-Ranch-gully1 | storm 1
Heil-Ranch-gully1 | storm 2
--- finished Heil-Ranch-gully1 ---
```

**(a)** 9 lines: `3 gullies × (2 storm lines + 1 "finished" line)`.

**(b)** Each `--- finished ---` line appears *after* that gully's inner loop has run completely, because that `print` is indented at the **outer** loop's level, so it runs once per gully.

**(c)** 12 lines. Once the "finished" line is indented under the inner loop, it runs on *every* inner pass: `3 × 2 × 2 = 12`. Same words, different indentation, different program.
</details>

---
## Part 2 : why put a loop inside a loop?

In Lab 3, Exercise 10, you swept relative wetness (Rw) from 0.0 to 1.0 to see how a soil's stability changes as a storm soaks in. You did it for *one* soil. But the interesting question is:

#### How does wetness change stability in *each* of our soil scenarios?

Let's start with what you can already do: a sweep of the `"medium"` scenario over three Rw values. Notice how we hand the scenario to the function: `**params` unpacks the dictionary (Lecture 10), and `rw=rw` adds the one keyword argument that changes each pass.

In [7]:
rw_values = [0.0, 0.5, 1.0]

scenario = "medium"
params = soil_scenarios[scenario]

for rw in rw_values:
    fs = factor_of_safety(**params, rw=rw)
    print(f"{scenario:>7} | Rw = {rw:.1f} | FS = {fs:.3f} | {interpret_stability(fs)}")

 medium | Rw = 0.0 | FS = 2.365 | Stable
 medium | Rw = 0.5 | FS = 2.030 | Stable
 medium | Rw = 1.0 | FS = 1.695 | Stable


<div class="alert alert-success">

**Tip**: you can mix `**` unpacking with ordinary keyword arguments in the same call, **as long as the keyword isn't also inside the dictionary.** That's exactly why we removed `rw` from `soil_scenarios`. If `"rw"` were still in there, Python would raise `TypeError: ... got multiple values for keyword argument 'rw'`.

</div>

Now do the same for the `"high"` scenario. Without a nested loop, you'd copy, paste, and edit:

In [8]:
scenario = "high"
params = soil_scenarios[scenario]

for rw in rw_values:
    fs = factor_of_safety(**params, rw=rw)
    print(f"{scenario:>7} | Rw = {rw:.1f} | FS = {fs:.3f} | {interpret_stability(fs)}")

   high | Rw = 0.0 | FS = 1.329 | Stable
   high | Rw = 0.5 | FS = 1.119 | Marginally stable
   high | Rw = 1.0 | FS = 0.910 | Unstable


That's two scenarios. All four would take **four** copies of that loop, and 40 scenarios would take forty. (You may recognize this feeling from Lecture 09.)

Look at what changed between the two copies: only *which scenario*. And "do the same thing for each item in a collection" is exactly what a `for` loop is for. So: **put the Rw sweep inside a loop over the scenarios.**

In [9]:
for scenario, params in soil_scenarios.items():      # OUTER loop: one pass per scenario
    for rw in rw_values:                              # INNER loop: one pass per Rw value
        fs = factor_of_safety(**params, rw=rw)
        print(f"{scenario:>7} | Rw = {rw:.1f} | FS = {fs:.3f} | {interpret_stability(fs)}")

    low | Rw = 0.0 | FS = 5.330 | Stable
    low | Rw = 0.5 | FS = 3.997 | Stable
    low | Rw = 1.0 | FS = 2.665 | Stable
 medium | Rw = 0.0 | FS = 2.365 | Stable
 medium | Rw = 0.5 | FS = 2.030 | Stable
 medium | Rw = 1.0 | FS = 1.695 | Stable
   high | Rw = 0.0 | FS = 1.329 | Stable
   high | Rw = 0.5 | FS = 1.119 | Marginally stable
   high | Rw = 1.0 | FS = 0.910 | Unstable
extreme | Rw = 0.0 | FS = 1.460 | Stable
extreme | Rw = 0.5 | FS = 1.227 | Marginally stable
extreme | Rw = 1.0 | FS = 0.994 | Unstable


#### What just happened?
1. The **outer** loop took its first scenario, `"low"`.
2. The **inner** loop then ran *completely*: `0.0`, `0.5`, `1.0`.
3. Only then did the outer loop take its second scenario, `"medium"`, and the inner loop **started over from the beginning**.
4. ...and so on until the outer loop ran out of scenarios.

These are the same three rules from the `i`/`j` table in Part 1, just with real data. The line inside the inner loop ran `4 scenarios × 3 Rw values = 12` times.

---
## Part 3 : looping through nested data

You've worked with nested data since Lecture 05 (and built a nested dictionary in Lab 2, Exercise 16): `gully_profiles` is a dictionary whose **values are themselves dictionaries**. Nested loops are the natural tool for walking through nested data:
- the **outer** loop goes through the outer structure (each gully)
- the **inner** loop goes through what's *inside* each item (each gully's properties)

In [10]:
for gully, props in gully_profiles.items():
    for key, value in props.items():
        print(f"{gully:>18} | {key:<21} | {value}")

 Heil-Ranch-gully1 | area_km2              | 0.1538
 Heil-Ranch-gully1 | burn_severity         | severe
 Heil-Ranch-gully1 | normal_infiltration   | 16
 Heil-Ranch-gully1 | postfire_infiltration | 2.25
 Heil-Ranch-gully2 | area_km2              | 0.2523
 Heil-Ranch-gully2 | burn_severity         | moderate
 Heil-Ranch-gully2 | normal_infiltration   | 16
 Heil-Ranch-gully2 | postfire_infiltration | 11.77
       Geer-Canyon | area_km2              | 7.325
       Geer-Canyon | burn_severity         | severe
       Geer-Canyon | normal_infiltration   | 22
       Geer-Canyon | postfire_infiltration | 3.5
     Central-Gulch | area_km2              | 0.1515
     Central-Gulch | burn_severity         | moderate
     Central-Gulch | normal_infiltration   | 16
     Central-Gulch | postfire_infiltration | 4.0


#### Let's walk through the first few passes
| **outer pass** | **`gully`** | **`props`** | **inner pass** | **`key`** | **`value`** |
|-:|:-|:-|-:|:-|-:|
| 1 | `"Heil-Ranch-gully1"` | `{"area_km2": 0.1538, "burn_severity": "severe", ...}` | 1 | `"area_km2"` | `0.1538` |
| 1 | `"Heil-Ranch-gully1"` | *(same dictionary)* | 2 | `"burn_severity"` | `"severe"` |
| 1 | `"Heil-Ranch-gully1"` | *(same dictionary)* | 3 | `"normal_infiltration"` | `16` |
| 1 | `"Heil-Ranch-gully1"` | *(same dictionary)* | 4 | `"postfire_infiltration"` | `2.25` |
| 2 | `"Heil-Ranch-gully2"` | `{"area_km2": 0.2523, ...}` | 1 | `"area_km2"` | `0.2523` |

Notice that the inner loop's collection, `props.items()`, **depends on the outer loop's variable**, `props`. That's the signature of a nested loop through nested data: whatever the outer loop hands you *becomes the thing the inner loop walks through.* And since `props` is a dictionary, the inner loop uses `.items()` and unpacks two variables, exactly like Lecture 10.

<div class="alert alert-danger">

**Common error**: forgetting `.items()` on the inner loop

What happens if we write two loop variables but forget `.items()`?

</div>

In [11]:
for gully, props in gully_profiles.items():
    for key, value in props:
        print(f"{gully:>18} | {key:<21} | {value}")

ValueError: too many values to unpack (expected 2)

Looping over a dictionary directly gives you only its **keys**, which are strings like `"area_km2"`. Python then tries to squeeze that 8-character string into just two variables (`key` and `value`) and can't: `too many values to unpack (expected 2)`.

**Whenever you see "too many values to unpack"**, check that every loop with two loop variables has a partner that produces pairs: `.items()`, `zip()`, or `enumerate()`.

<div class="alert alert-info">

### Exercise 2
Use a nested loop to print only each gully's `burn_severity` and `postfire_infiltration` (skip the other two), formatted like `Geer-Canyon | burn_severity = severe`.

*Hint: in Lecture 05 you learned an operator that checks whether a value is `in` a list.*

</div>

In [ ]:
# your answer here
wanted = ["burn_severity", "postfire_infiltration"]

for gully, props in gully_profiles.items():
    for key, value in _______:
        if key ___ wanted:
            print(f"{gully:>18} | {key} = {value}")

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
wanted = ["burn_severity", "postfire_infiltration"]

for gully, props in gully_profiles.items():
    for key, value in props.items():
        if key in wanted:
            print(f"{gully:>18} | {key} = {value}")
```
---
```
 Heil-Ranch-gully1 | burn_severity = severe
 Heil-Ranch-gully1 | postfire_infiltration = 2.25
 Heil-Ranch-gully2 | burn_severity = moderate
 Heil-Ranch-gully2 | postfire_infiltration = 11.77
       Geer-Canyon | burn_severity = severe
       Geer-Canyon | postfire_infiltration = 3.5
     Central-Gulch | burn_severity = moderate
     Central-Gulch | postfire_infiltration = 4.0
```
</details>

---
## Part 4 : building results with a nested loop

So far our nested loops only *print*. In real analysis you want to **store** results so you can compare them, plot them, or hand them to another function. Here's a pattern you'll use constantly: **one list of results per scenario, stored in a dictionary keyed by the scenario's name.**

First, the full Rw sweep from Lab 3. We'll use `range()` and the integer-counting trick from Lecture 10 to get `0.0, 0.1, ..., 1.0` with no floating-point drift:

In [12]:
rw_values = []                        # start with an empty list (this *reassigns* rw_values)

for i in range(11):
    rw_values.append(i / 10)          # count in whole numbers, divide fresh each time

print(rw_values)

[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]


#### The recipe
1. Make an **empty dictionary** for the results.
2. **Outer loop** over the scenarios.
3. **Inside the outer loop, but before the inner loop**: make a *fresh, empty list* for this scenario.
4. **Inner loop**: compute a value and `.append()` it to that list.
5. **After the inner loop** (still inside the outer loop): store the finished list in the dictionary under the scenario's name.

In [13]:
fs_by_scenario = {}                                       # 1. empty results dictionary

for scenario, params in soil_scenarios.items():           # 2. outer loop: one pass per scenario
    fs_list = []                                          # 3. a FRESH empty list for this scenario
    for rw in rw_values:                                  # 4. inner loop: one pass per Rw value
        fs = factor_of_safety(**params, rw=rw)
        fs_list.append(fs)
    fs_by_scenario[scenario] = fs_list                    # 5. store this scenario's finished list

print("scenarios stored  :", list(fs_by_scenario.keys()))
print("values per scenario:", len(fs_by_scenario["high"]))
print(fs_by_scenario["high"])

scenarios stored  : ['low', 'medium', 'high', 'extreme']
values per scenario: 11
[1.3292450044768238, 1.2872900229179598, 1.2453350413590958, 1.2033800598002318, 1.1614250782413678, 1.1194700966825037, 1.0775151151236397, 1.0355601335647757, 0.9936051520059117, 0.9516501704470478, 0.9096951888881837]


#### The shape of what we built: a dictionary of lists
```python
fs_by_scenario = {
    "low":     [FS at Rw=0.0, FS at Rw=0.1, FS at Rw=0.2, ..., FS at Rw=1.0],
    "medium":  [FS at Rw=0.0, FS at Rw=0.1, FS at Rw=0.2, ..., FS at Rw=1.0],
    "high":    [FS at Rw=0.0, FS at Rw=0.1, FS at Rw=0.2, ..., FS at Rw=1.0],
    "extreme": [FS at Rw=0.0, FS at Rw=0.1, FS at Rw=0.2, ..., FS at Rw=1.0],
}
```
Every **key** came from the outer loop, and every **position in a list** came from the inner loop. Position `i` of every list matches `rw_values[i]`.

#### Reading it back
A dictionary of lists is easy to summarize. Python has built-in functions that work on any list: `min()`, `max()`, `sum()`, and `len()`.

In [14]:
for scenario, fs_list in fs_by_scenario.items():
    average = sum(fs_list) / len(fs_list)
    print(f"{scenario:>7}: min FS = {min(fs_list):.3f} | max FS = {max(fs_list):.3f} | mean FS = {average:.3f}")

    low: min FS = 2.665 | max FS = 5.330 | mean FS = 3.997
 medium: min FS = 1.695 | max FS = 2.365 | mean FS = 2.030
   high: min FS = 0.910 | max FS = 1.329 | mean FS = 1.119
extreme: min FS = 0.994 | max FS = 1.460 | mean FS = 1.227


<div class="alert alert-success">

**Tip**: `statistics.mean(fs_list)` from Lecture 07 gives the same answer. `sum(fs_list) / len(fs_list)` is still worth knowing because it needs no `import`.

**Another tip**: `max()` also works on separate values: `max(0, -3.2)` gives `0`, and `max(0, 4.7)` gives `4.7`. It's a compact way to say "never let this go below zero," which we'll use in a minute.

</div>

And when you need to look *inside* each list, nest another loop. Here we **count** how many Rw values push each scenario into the unstable range (FS < 1). Notice `zip()` from Lecture 10 in the inner loop:

In [15]:
for scenario, fs_list in fs_by_scenario.items():
    n_unstable = 0                                        # a FRESH counter for each scenario
    for rw, fs in zip(rw_values, fs_list):
        if fs < 1.0:
            n_unstable += 1
    print(f"{scenario:>7}: {n_unstable:>2} of {len(fs_list)} Rw values give FS < 1 (unstable)")

    low:  0 of 11 Rw values give FS < 1 (unstable)
 medium:  0 of 11 Rw values give FS < 1 (unstable)
   high:  3 of 11 Rw values give FS < 1 (unstable)
extreme:  1 of 11 Rw values give FS < 1 (unstable)


<div class="alert alert-success">

**Tip**: notice that `n_unstable = 0` lives in the same spot as `fs_list = []` did: **inside the outer loop, before the inner loop.** Any list, counter, or running total that should start fresh *for each outer item* gets created there.

</div>

### Now the Calwood gullies
Same pattern, different science. In Lab 2, Exercise 10, you calculated the runoff from a storm at Geer Canyon and Central Gulch by hand. Here's the workflow, for **every gully** and **every storm**:

1. Runoff rate: `Q = intensity − ET − postfire_infiltration` (mm/hr). If a storm is weaker than the soil can absorb, `Q` would come out negative, but runoff can't be negative, so we clamp it with `max(0, ...)`.
2. Runoff volume (m$^3$): `Q × area_km2 × 1000 × duration`. (The `1000` is a unit conversion: 1 mm of runoff spread over 1 km$^2$ is $10^{-3}$ m × $10^{6}$ m$^2$ = 1000 m$^3$.)

Notice the storms are stored as **tuples**, `(intensity, duration)`, so the inner loop can unpack them right on the `for` line, just like `zip()` did.

<div class="alert alert-info">

### Exercise 3 : runoff for every gully and every storm
Build a dictionary called `runoff_volumes` with one list per gully, holding the **post-fire** runoff volume (m$^3$) for each of the three `storms`. Use the recipe: outer loop over `gully_profiles`, a fresh list for each gully, inner loop over `storms`.

The last two lines print Geer Canyon's three volumes. *(Format tip: `{volume:,.0f}` adds thousands separators and rounds to a whole number.)*

</div>

In [ ]:
# your answer here
runoff_volumes = {}

for gully, props in gully_profiles.items():
    volumes = _____
    for intensity, duration in _____:
        Q = _____(0, intensity - ET - props["postfire_infiltration"])     # runoff rate (mm/hr), never negative
        volume = Q * props["_____"] * 1000 * duration                       # m^3
        volumes.append(volume)
    runoff_volumes[gully] = _____

for storm_number, volume in enumerate(runoff_volumes["Geer-Canyon"]):
    print(f"Geer-Canyon | storm {storm_number + 1} | runoff volume = {volume:,.0f} m^3")

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
runoff_volumes = {}

for gully, props in gully_profiles.items():
    volumes = []
    for intensity, duration in storms:
        Q = max(0, intensity - ET - props["postfire_infiltration"])     # runoff rate (mm/hr), never negative
        volume = Q * props["area_km2"] * 1000 * duration                 # m^3
        volumes.append(volume)
    runoff_volumes[gully] = volumes

for storm_number, volume in enumerate(runoff_volumes["Geer-Canyon"]):
    print(f"Geer-Canyon | storm {storm_number + 1} | runoff volume = {volume:,.0f} m^3")
```
---
```
Geer-Canyon | storm 1 | runoff volume = 172,138 m^3
Geer-Canyon | storm 2 | runoff volume = 16,848 m^3
Geer-Canyon | storm 3 | runoff volume = 846,038 m^3
```
</details>

<div class="alert alert-warning">

### Reflection
Geer Canyon produces far more runoff *volume* than the other gullies. Is that because its soil is worse, or because of something else? Check by comparing the runoff **rate** `Q` (mm/hr, which doesn't depend on basin area) for Storm 1 across the four gullies. What stands out about `Heil-Ranch-gully2`?

</div>

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

Geer Canyon's volume is huge mostly because its basin is huge: about 7.3 km$^2$, roughly 30 to 48 times larger than the other three gullies (0.15 to 0.25 km$^2$). Per unit area, its Storm 1 runoff rate (30 − 3 − 3.5 = **23.5 mm/hr**) is nearly the same as `Heil-Ranch-gully1` (24.75 mm/hr) and `Central-Gulch` (23.0 mm/hr).

`Heil-Ranch-gully2` stands out: only **15.2 mm/hr**, far lower than the others. Its post-fire infiltration is 11.77 mm/hr, versus 2 to 4 mm/hr in the other three, so much more of the storm soaks in, and less runs off. (Data from Lab 2.)
</details>

---
## Part 5 : three classic nested-loop bugs

Nested loops give you more places to make mistakes. Two habits catch most of them:
1. **Shrink the problem.** Debug with 2 scenarios × 2 Rw values, not 4 × 11.
2. **Print the loop variables** on the first line inside each loop, so you can *see* the loop step.

Here are the three you're most likely to hit.

<div class="alert alert-danger">

### Bug 1 : forgetting to reset the accumulator
What if we create the empty list **once**, outside both loops?

</div>

In [16]:
fs_bad = []                                           # <-- created ONCE, outside both loops
fs_by_scenario_bad = {}

for scenario, params in soil_scenarios.items():
    for rw in rw_values:
        fs_bad.append(factor_of_safety(**params, rw=rw))
    fs_by_scenario_bad[scenario] = fs_bad

print("values stored for 'low':", len(fs_by_scenario_bad["low"]))
print("do 'low' and 'high' share one list?", fs_by_scenario_bad["low"] is fs_by_scenario_bad["high"])

values stored for 'low': 44
do 'low' and 'high' share one list? True


**No error was raised**, and that's what makes this one dangerous. One list collected *every* scenario's values (`4 × 11 = 44`), and every key in the dictionary points to that **same** list. The `is` operator asks whether two names point at the very same object, and here the answer is `True`: this is called **aliasing** (two names, one list). It's the same thing you met in the Week 2 worksheet, and the reason `.copy()` exists.

**The fix:** create the list *inside the outer loop*, before the inner loop, so each scenario gets its own.

<div class="alert alert-danger">

### Bug 2 : reusing a loop variable name
What if the inner loop reuses the outer loop's variable name?

</div>

In [17]:
for scenario, params in soil_scenarios.items():
    for scenario in rw_values:                        # oops: `scenario` reused as the inner loop variable
        fs = factor_of_safety(**params, rw=scenario)
    print(f"Finished the Rw sweep for {scenario}")

Finished the Rw sweep for 1.0
Finished the Rw sweep for 1.0
Finished the Rw sweep for 1.0
Finished the Rw sweep for 1.0


Again, **no error.** The inner loop *overwrote* `scenario` (a scenario name) with Rw values, so after the inner loop finished, `scenario` was `1.0`, the last Rw value. This is called **shadowing**: the second variable hides the first.

**The fix:** give every loop variable a name that says what it holds: `scenario`, `rw`, `gully`, `storm_number`. Printing the loop variables (habit #2 above) makes this bug jump out.

<div class="alert alert-danger">

### Bug 3 : doing the work in the wrong place
The third bug is about *indentation*, and you get to find it in the next exercise.

</div>

<div class="alert alert-info">

### Exercise 4 : find the bug
The code below is supposed to store **3** runoff volumes per gully (one per storm) in `volumes_fixed`. Run it. What's wrong, and why? Fix it so `len(volumes_fixed["Geer-Canyon"])` gives `3`.

</div>

In [ ]:
# find and fix the bug
volumes_fixed = {}

for gully, props in gully_profiles.items():
    volumes = []
    for intensity, duration in storms:
        Q = max(0, intensity - ET - props["postfire_infiltration"])
        volume = Q * props["area_km2"] * 1000 * duration
    volumes.append(volume)
    volumes_fixed[gully] = volumes

print(len(volumes_fixed["Geer-Canyon"]))     # should be 3
print(volumes_fixed["Geer-Canyon"])

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
volumes_fixed = {}

for gully, props in gully_profiles.items():
    volumes = []
    for intensity, duration in storms:
        Q = max(0, intensity - ET - props["postfire_infiltration"])
        volume = Q * props["area_km2"] * 1000 * duration
        volumes.append(volume)              # indented one more level: runs on EVERY inner pass
    volumes_fixed[gully] = volumes

print(len(volumes_fixed["Geer-Canyon"]))     # should be 3
print(volumes_fixed["Geer-Canyon"])
```
---
```
3
[172137.50000000003, 16847.5, 846037.5]
```

`volumes.append(volume)` was indented at the **outer** loop's level, so it ran once per gully, *after* the inner loop had finished. By then `volume` only remembered the **last** storm's value (Storm 3), so each list held a single number. Moving `.append()` one level deeper puts it inside the inner loop, where it runs on every pass.
</details>

---
## Part 6 : a first look at `continue` and `break`

Sometimes you don't want every pass through a loop to do the same thing. Two keywords change the flow. We'll spend Friday on them; today, one example of each inside a nested loop.

| **keyword** | **what it does** |
|:-|:-|
| `continue` | skip the rest of **this pass** and jump to the next pass of the loop |
| `break` | **exit the loop** it sits in, right now |

### `continue`: skip storms that make no runoff
Under **normal** (unburned) infiltration, a weak storm may be entirely absorbed by the soil, so there's no runoff to report. Let's skip those. (This also uses `enumerate()` from Lecture 10 to number the storms.)

In [18]:
for gully, props in gully_profiles.items():
    for storm_number, (intensity, duration) in enumerate(storms):
        Q = intensity - ET - props["normal_infiltration"]
        if Q <= 0:
            continue                                   # all the rain soaks in: skip the rest of this pass
        print(f"{gully:>18} | storm {storm_number + 1} | runoff = {Q:5.1f} mm/hr")

 Heil-Ranch-gully1 | storm 1 | runoff =  11.0 mm/hr
 Heil-Ranch-gully1 | storm 3 | runoff =  26.0 mm/hr
 Heil-Ranch-gully2 | storm 1 | runoff =  11.0 mm/hr
 Heil-Ranch-gully2 | storm 3 | runoff =  26.0 mm/hr
       Geer-Canyon | storm 1 | runoff =   5.0 mm/hr
       Geer-Canyon | storm 3 | runoff =  20.0 mm/hr
     Central-Gulch | storm 1 | runoff =  11.0 mm/hr
     Central-Gulch | storm 3 | runoff =  26.0 mm/hr


Storm 2 never appears: for every gully, `Q` came out zero or negative, so `continue` skipped the `print`. Now the **exact same loop**, with one word changed: `normal_infiltration` becomes `postfire_infiltration`.

In [19]:
for gully, props in gully_profiles.items():
    for storm_number, (intensity, duration) in enumerate(storms):
        Q = intensity - ET - props["postfire_infiltration"]
        if Q <= 0:
            continue
        print(f"{gully:>18} | storm {storm_number + 1} | runoff = {Q:5.1f} mm/hr")

 Heil-Ranch-gully1 | storm 1 | runoff =  24.8 mm/hr
 Heil-Ranch-gully1 | storm 2 | runoff =  12.8 mm/hr
 Heil-Ranch-gully1 | storm 3 | runoff =  39.8 mm/hr
 Heil-Ranch-gully2 | storm 1 | runoff =  15.2 mm/hr
 Heil-Ranch-gully2 | storm 2 | runoff =   3.2 mm/hr
 Heil-Ranch-gully2 | storm 3 | runoff =  30.2 mm/hr
       Geer-Canyon | storm 1 | runoff =  23.5 mm/hr
       Geer-Canyon | storm 2 | runoff =  11.5 mm/hr
       Geer-Canyon | storm 3 | runoff =  38.5 mm/hr
     Central-Gulch | storm 1 | runoff =  23.0 mm/hr
     Central-Gulch | storm 2 | runoff =  11.0 mm/hr
     Central-Gulch | storm 3 | runoff =  38.0 mm/hr


After the fire, **every** storm, even the weak Storm 2, produces runoff in every gully. That's the fire's effect on infiltration (Lab 2), made visible by two nested loops and a `continue`.

### `break`: stop when we find it
For each soil scenario, at what Rw does the Factor of Safety **first** drop below 1 (unstable)? Once we've found it, there's no reason to keep checking wetter conditions.

In [20]:
for scenario, params in soil_scenarios.items():
    for rw in rw_values:
        fs = factor_of_safety(**params, rw=rw)
        if fs < 1.0:
            print(f"{scenario:>7}: first unstable at Rw = {rw:.1f} (FS = {fs:.3f})")
            break                                      # stop the INNER loop for this scenario

   high: first unstable at Rw = 0.8 (FS = 0.994)
extreme: first unstable at Rw = 1.0 (FS = 0.994)


<div class="alert alert-danger">

**Common mistake**: `break` does not end *everything*

`break` exits only the **innermost** loop it sits in. Notice the output above has a line for `high` **and** for `extreme`: after `break` ended `high`'s inner loop, the **outer** loop carried on to `extreme`.

</div>

Also notice who's *missing*: `low` and `medium` printed nothing. They never dropped below FS = 1 anywhere in the sweep, so the `if` was never `True` and `break` never ran.

<div class="alert alert-info">

### Exercise 5
**(a)** Modify the `break` loop above to find, for each scenario, the first Rw at which FS drops below **1.3** (the boundary of the "Marginally stable" category from `interpret_stability()`). Print the scenario, the Rw, and the FS, then `break`. **Which scenarios print nothing? Why?**

**(b)** Modify the *post-fire* `continue` loop above so it only prints gully–storm pairs with a runoff rate of **20 mm/hr or more**, skipping the rest with `continue`. (20 mm/hr is an arbitrary cutoff for this exercise.) How many lines print?

</div>

In [ ]:
# (a) your answer here
for scenario, params in soil_scenarios.items():
    for rw in rw_values:
        fs = factor_of_safety(**params, rw=rw)
        if fs _ 1.3:
            print(f"{scenario:>7}: first below FS = 1.3 at Rw = {rw:.1f} (FS = {fs:.3f})")
            _____

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
for scenario, params in soil_scenarios.items():
    for rw in rw_values:
        fs = factor_of_safety(**params, rw=rw)
        if fs < 1.3:
            print(f"{scenario:>7}: first below FS = 1.3 at Rw = {rw:.1f} (FS = {fs:.3f})")
            break
```
---
```
   high: first below FS = 1.3 at Rw = 0.1 (FS = 1.287)
extreme: first below FS = 1.3 at Rw = 0.4 (FS = 1.273)
```

**`low` and `medium` print nothing.** Even fully saturated (Rw = 1.0), their lowest FS values (2.665 and 1.695) stay well above 1.3, so `fs < 1.3` was never `True`, the loop ran all the way to the end without hitting `break`, and nothing got printed. The loop can't tell us "this scenario never got that low" on its own. On Friday we'll learn how to detect that case.
</details>

In [ ]:
# (b) your answer here
for gully, props in gully_profiles.items():
    for storm_number, (intensity, duration) in enumerate(storms):
        Q = intensity - ET - props["postfire_infiltration"]
        if Q _ 20:
            _____
        print(f"{gully:>18} | storm {storm_number + 1} | runoff = {Q:5.1f} mm/hr")

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
for gully, props in gully_profiles.items():
    for storm_number, (intensity, duration) in enumerate(storms):
        Q = intensity - ET - props["postfire_infiltration"]
        if Q < 20:
            continue
        print(f"{gully:>18} | storm {storm_number + 1} | runoff = {Q:5.1f} mm/hr")
```
---
```
 Heil-Ranch-gully1 | storm 1 | runoff =  24.8 mm/hr
 Heil-Ranch-gully1 | storm 3 | runoff =  39.8 mm/hr
 Heil-Ranch-gully2 | storm 3 | runoff =  30.2 mm/hr
       Geer-Canyon | storm 1 | runoff =  23.5 mm/hr
       Geer-Canyon | storm 3 | runoff =  38.5 mm/hr
     Central-Gulch | storm 1 | runoff =  23.0 mm/hr
     Central-Gulch | storm 3 | runoff =  38.0 mm/hr
```

**7 lines.** Storm 2 never reaches 20 mm/hr in any gully, and `Heil-Ranch-gully2` only reaches it in Storm 3, because its infiltration has recovered to 11.77 mm/hr.
</details>

---
## Wrap-up

| **idea** | **remember** |
|:-|:-|
| how a nested loop runs | outer takes **one** step → inner runs **all the way through** → outer takes the next step → inner **restarts** |
| rows and columns | outer loop = down the rows, inner loop = across the columns |
| how many passes | `(outer passes) × (inner passes)` |
| nested data | the inner loop's collection comes from the outer loop's variable (`props.items()`) |
| building results | a fresh list/counter goes **inside the outer loop, before the inner loop** |
| naming | a **different, descriptive name** for every loop variable |
| `continue` | skip the rest of *this pass* |
| `break` | exit the *innermost* loop only |
| debugging | shrink the problem; print the loop variables |

**Wednesday:** more nested loops, on new data: loops whose inner range depends on the outer loop, building tables, and a first look at how loops make plotting easy.

**Friday:** `break` and `continue` in depth: how to tell when a loop *finished without* finding anything, and how to keep a "best so far" while looping.

---
## Extra Practice

<div class="alert alert-info">

### Exercise 6 : a debris-flow threshold map
The Caine (1980) threshold says a storm of duration $D$ is likely to trigger a debris flow when its intensity is at least $I = 14.82D^{-0.39}$ (that's our `caine_threshold()` function). Let's map it: for every combination of storm duration (rows) and rainfall intensity (columns), print an `X` if a debris flow is predicted and a `.` if not.

The header line is done for you. Fill in the loops. Remember: the **outer** loop walks down the rows, the **inner** loop walks across the columns, and the bare `print()` goes at the **outer** loop's level.

<span style="font-size:0.85em;color:#6b7a8d;"><em>Source: Caine, N. (1980). The rainfall intensity-duration control of shallow landslides and debris flows. Geografiska Annaler: Series A, 62(1-2), 23-27.</em></span>

</div>

In [ ]:
# your answer here
durations = [0.25, 0.5, 1, 2, 3, 6, 12, 24]        # hours
intensities = [5, 10, 15, 20, 25, 30, 40, 50]      # mm/hr

print(f"{'I (mm/hr)':<14}", end="")                 # header line (done for you)
for intensity in intensities:
    print(f"{intensity:>3}", end=" ")
print()

for duration in _____:
    print(f"D = {duration:>5} hr  ", end="")
    for intensity in _____:
        if intensity >= caine_threshold(_____):
            symbol = "X"
        else:
            symbol = "."
        print(f"{symbol:>3}", end=" ")
    print()

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
durations = [0.25, 0.5, 1, 2, 3, 6, 12, 24]        # hours
intensities = [5, 10, 15, 20, 25, 30, 40, 50]      # mm/hr

print(f"{'I (mm/hr)':<14}", end="")                 # header line (done for you)
for intensity in intensities:
    print(f"{intensity:>3}", end=" ")
print()

for duration in durations:
    print(f"D = {duration:>5} hr  ", end="")
    for intensity in intensities:
        if intensity >= caine_threshold(duration):
            symbol = "X"
        else:
            symbol = "."
        print(f"{symbol:>3}", end=" ")
    print()
```
---
```
I (mm/hr)       5  10  15  20  25  30  40  50 
D =  0.25 hr    .   .   .   .   .   X   X   X 
D =   0.5 hr    .   .   .   X   X   X   X   X 
D =     1 hr    .   .   X   X   X   X   X   X 
D =     2 hr    .   .   X   X   X   X   X   X 
D =     3 hr    .   X   X   X   X   X   X   X 
D =     6 hr    .   X   X   X   X   X   X   X 
D =    12 hr    .   X   X   X   X   X   X   X 
D =    24 hr    X   X   X   X   X   X   X   X 
```

Read down the columns: the shorter the storm, the more intense it has to be to cross the threshold, so the `X`s form a staircase. A 24-hour storm at just 5 mm/hr is already over the line, while a 15-minute storm needs about 25.5 mm/hr.
</details>

<div class="alert alert-custom">

### Exercise 7 (bonus) : the same FS data as a list of lists
Sometimes data arrives as a **list of lists** instead of a dictionary of lists. Here is `fs_by_scenario` converted:

```python
scenario_names = list(fs_by_scenario.keys())
grid = list(fs_by_scenario.values())       # rows = scenarios, columns = Rw values
```

Use **nested `range(len(...))` loops** and double indexing (`grid[row][col]`) to print every scenario/Rw combination where FS is below **1.0** (unstable). Use `scenario_names[row]` and `rw_values[col]` to label each line.

</div>

In [ ]:
# your answer here
scenario_names = list(fs_by_scenario.keys())
grid = list(fs_by_scenario.values())

for row in range(len(_____)):
    for col in range(len(_____)):
        if grid[row][col] _ 1.0:
            print(f"{scenario_names[____]:>7} | Rw = {rw_values[____]:.1f} | FS = {grid[row][col]:.3f}")

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
scenario_names = list(fs_by_scenario.keys())
grid = list(fs_by_scenario.values())

for row in range(len(grid)):
    for col in range(len(grid[row])):
        if grid[row][col] < 1.0:
            print(f"{scenario_names[row]:>7} | Rw = {rw_values[col]:.1f} | FS = {grid[row][col]:.3f}")
```
---
```
   high | Rw = 0.8 | FS = 0.994
   high | Rw = 0.9 | FS = 0.952
   high | Rw = 1.0 | FS = 0.910
extreme | Rw = 1.0 | FS = 0.994
```
</details>